# RINGSS Monochrome Simulation

Auth: Diego H.

We will try to recreate the simulation code written in IDL. \
This ipynb will act as the testsimul.pro counterpart.\
\
While at first, simatm.pro, ringsim.pro and other files will be on cells, \
these should be written in their own .py scripts. 

In [ ]:
import logging
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scipy as sci
from tqdm.notebook import tqdm

from astropy.io import fits
import datetime
from IPython.display import display, clear_output
from scipy.signal import detrend, find_peaks
from scipy.ndimage import zoom
from scipy.special import factorial

logger = logging.getLogger("Simulation")
logger.setLevel("DEBUG")

In [ ]:
# Parameters, these can/should be read from a .json parameter file

d = 0.304             # meters, M1
eps = 0.7             # %, central obscuration
pixel = 1.09          # arcsec, pixel scale (supposedly)
pixel_um = 6.9        # microns, camera pixel size
focal_length = 1.3023 # meters, telescope effective focal length
lamb = 0.6         # meters, wavelength
pdist = 924           # meters, conjugation distance
ringradpix = 7.5      # pixels, inner ring radius
drad = 1.5            # lamb/(0.5*D*(1-eps)) units, mask width
mmax = 20             # maximum order of angular signals
nsect = 8             # number of sectors for radius calculation
interpol = 1          # sub-pixel shifts
ron = 0               # electrons, camera read out noise
roi_size = 32         # pixels, region of interest to be used

asperpix = (pixel_um * 1e-6 / focal_length) * 206265  # Derived angular scale in arcseconds
pixel_m = asperpix / 206265 * pdist                   # Projected physical pixel size at conjugation height

seeing = 1     # seeing in arcsec at 0.6 micron
zlow = 500       # lower layer altitude [m]
zhigh = 11000    # higher layer altitude
fhigh = 0.1      # Also highfrac
starmag = 2      # star magnitude

see_rad = seeing / 206265  # Convert to radians
r0 = 0.98 * lamb / see_rad  # Fried parameter

# The turbulence integral J (tint)
# We calculate it using the standard 500nm reference to keep 'tint' scale-invariant
lamb_ref = 0.5e-6
r0_ref = r0 * (lamb_ref / lamb)**(1.2) # Scaling r0 to 500nm
tint = (r0_ref**(-5/3)/0.423)*(0.5*lamb_ref/np.pi)**2
# Corrected 'see' calculation for 600nm
see = 206265 * 0.98 * lamb * ((tint * 0.423) / (0.5 * lamb / np.pi) ** 2) ** (0.6)

logger.debug(f'r0: {r0}, tint: {tint}, see: {see}, seeing: {seeing}')
logger.debug(f'pixel: {pixel}, asperpix: {asperpix}, pixel_um: {pixel_um}, pixel_m: {pixel_m}')

# simatm.pro

def simatm(ngrid,pixel,lamb,r0,fhigh,zlow,zhigh,seed0=seed0):

In [ ]:
ngrid = 1024
pixel = 0.0176640  # idek why
lamb = lamb
r0 = r0
zlow = zlow
zhigh = zhigh
fhigh = fhigh

In [ ]:
print('Atmosphere simulation, parameters calculated given below.\n')
if 'seed0' in globals() and seed0 is not None:       # If seed0 is provided, use it. Otherwise, let the OS provide a random one
    rng = np.random.default_rng(seed0)
    logger.debug(f'Simulation on fixed seed: {seed0}.')
else:
    rng = np.random.default_rng()
    logger.debug('Simulation on random seed.')

# Check for consistency
if zlow > zhigh:
    raise ValueError(f"Inconsistency: zlow ({zlow}) can't be greater than zhigh ({zhigh})")
    #return

size = 2 * ngrid * pixel                                 # 2 times the size of the chosen grid in arc-seconds or meters

# Starting parameters
lamb_ref = 0.5                                                # reference wavelength
r0_ref = r0 * (lamb_ref / lamb) ** (1.2)                            # scaling r0 to 500nm
tint0 = (r0_ref ** ( -5 / 3) / 0.423) * (0.5 * lamb_ref/ np.pi) ** 2            # correct total turbulence integral for given wavelength
tint1 = tint0 * fhigh                                               # high-layer integral
tint2 = tint0 * (1 - fhigh)                                          # low-layer integral
r01 = (0.423 *((0.5 * lamb /np.pi ) ** (-2)) * tint1) ** (-3 / 5)              # high fried parameter
r02 = (0.423 *((0.5 * lamb /np.pi ) ** (-2)) * tint2) ** (-3 / 5)              # low fried parameter

see = 206265 * 0.98 * lamb * ((tint0 * 0.423) / (0.5 * lamb / np.pi) ** 2) ** (0.6) # simulated seeing @ given wavelength

print(f'Grid size arcsec: {size} \n Fried parameters, meters (low, high): {r02, r01} \n',
            f'Turbulence integrals, m^1/3: {tint2,tint1} \n Altitudes, meters: {zlow, zhigh}\n',
            f'Starting seeing: {see} @ {lamb} microns\n')
logger.debug(f'tint0: {tint0}, r0: {r0}')


# Phase simulations
fact1 = np.sqrt(0.023)*(size/r01)**(5/6)
fact2 = np.sqrt(0.023)*(size/r02)**(5/6)

if fhigh == 0:
    fact1 = 0


# Coord grid for atmosphere
N = 2 * ngrid
x = np.linspace(-ngrid, ngrid - 1, N)
y = np.linspace(-ngrid, ngrid - 1, N)
xx, yy = np.meshgrid(x, y)
r = np.sqrt(xx**2 + yy**2)
r[ngrid, ngrid] = 1e-4 # Arbitrary near-zero value for non-zero division 


# Fresnel Filters
farg = (np.pi*lamb*r**2)/(size**2)

# Simulate the high layer
if fhigh > 0:
    logger.debug(f'fhigh = {fhigh}')
    print('Simulating high screen')
    
    noise_real = rng.standard_normal((2*ngrid, 2*ngrid))                # create the noise screens
    noise_imag = rng.standard_normal((2*ngrid, 2*ngrid))
    complex_noise = noise_real + 1j * noise_imag

    #with np.errstate(divide='ignore'):
    temp = fact1 * (r**(-11/6)) * complex_noise                         # kolmogorov Power Law (r^-11/6)
    temp[ngrid, ngrid] = 0.0 + 0.0j

    temp_spatial = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(temp)))# spatial domain (ifft2 is the 2D Inverse FFT)
    phase_screen = temp_spatial.real

    u1 = np.exp(1j * phase_screen)                                      # convert to Complex Amplitude


# Propagate to low layer, with Angular Spectrum method
if (zhigh-zlow) > 0:
    logger.debug(f'zhigh: {zhigh}, zlow:{zlow}, difference: {zhigh-zlow}')
    print('Propagating to low layer')

    u1_freq = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(u1)))    # transform the high layer to Freq Domain
    transfer_function = np.exp(-1j * farg * (zhigh-zlow))           # apply the Fresnel Transfer Function (farg earlier)
    temp = transfer_function * u1_freq
    u1 = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(temp)))      # back to Spatial domain

else:
    
    u1 = np.ones((2*ngrid, 2*ngrid), dtype=complex)                 # if no turbulence, the amplitude is flat (np.ones)
    print('No turbulence computed')

# Simulate turbulence in low layer
print('Simulating low layer')
noise_low = rng.standard_normal((2*ngrid, 2*ngrid)) + 1j * rng.standard_normal((2*ngrid, 2*ngrid))

#with np.errstate(divide='ignore'):
temp_low = fact2 * (r**(-11/6)) * noise_low
temp_low[ngrid, ngrid] = 0.0 + 0.0j

phase_low = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(temp_low))).real # transform to Spatial domain
u1 *= np.exp(1j * phase_low)                                               # acummulate layers

# Propagate to ground
print('Propagating to ground \n')
u1_freq = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(u1)))
transfer_ground = np.exp(-1j * farg * zlow)                                # propagate over zlow
tmp_ground = transfer_ground * u1_freq

u1 = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(tmp_ground)))           # Back to Spatial domain

# Rytov number
intensity = np.abs(u1)**2                                                  # Scintillation
scind = np.mean((intensity - 1.0)**2)                                      # Index
rytov = 19.22 * (lamb**(-7/6)) * (zlow**(5/6) * tint2 + zhigh**(5/6) * tint1)

print(f"Rytov variance (Theoretical): {rytov}")
print(f"Scintillation Index (Simulated): {scind}\n")

np.savez('atm_simulation.npz', 
         u1=u1, ngrid=ngrid, pixel=pixel, lamb=lamb, 
         see=see, r0=r0, fhigh=fhigh, 
         zhigh=zhigh, zlow=zlow)

plt.figure(figsize=(6, 5))
plt.imshow(intensity, cmap='gray')
plt.title('Simulated atmosphere')
plt.colorbar(label='Intensity')
plt.show()

# ringsim.pro
def(parameters, cube, zn=zn, zrad=zrad, starmag=starmag, ron=ron, wind=wind, blur=blur, debug=debug, jitter=jitter):

In [ ]:
# Restore the Atmosphere
with np.load('atm_simulation.npz') as data:
    u1     = data['u1']
    ngrid  = data['ngrid']
    pixel  = data['pixel']
    lamb   = data['lamb']
    see    = data['see']
    r0     = data['r0']
    fhigh  = data['fhigh']
    zlow   = data['zlow']
    zhigh  = data['zhigh']

# Hard-coded Parameters
starmag = starmag if 'starmag' in locals() else 2.0 # star magnitude
ron = ron if 'ron' in locals() else 1.0             # electrons, read out noise 
d = d                                               # meters, mirror diameter
eps = eps                                           # central obscuration 
pdist = pdist                                       # meters, H (Conjugation distance)
texp = 0.001                                        # seconds, exposure time
tacc = 2                                            # seconds, accumulation time
wind = wind if 'wind' in locals() else 10           # m/s, wind speed
grid_size = 2*ngrid                                 # ngrid is half-size
size = 2*ngrid*pixel                                # grid size in arcseconds
nap = ngrid                                         # size of aperture and image arrays in fine pixels
oversamp = True                                     # check to oversample

In [ ]:
# Atmospheric Blurring (Exposure Smearing)
if 'blur' in locals() and blur:
    nblur = int(wind * texp / pixel + 0.5)          # How many pixels the wind moves on 1 exposure
    print(f"Averaging atmospheric screens, N={nblur}")
    
    if nblur > 1:
        tmp = u1.copy()
        for k in range(1, nblur):
            tmp += np.roll(u1, k, axis=1)           # GDL's shift along the x-axis
        u1 = tmp / nblur

# Physical Constants and Seeing Calculation
if 'zlow' not in locals() or zlow is None:
    zlow = zhigh

seeing_arcsec = 0.98 * lamb / r0 * 206265 # Seeing (FWHM) in arcseconds
print(f"Seeing: {seeing_arcsec} arcsec")
logger.debug(f'Seeing from simatm: {see}')
print(f"Layers at: zlow={zlow}m, zhigh={zhigh}m. High fraction: {fhigh}")
print(f"Total screen size: {2*ngrid*pixel} arcsec, Pixel size: {pixel} arcsec")

# Turbulence Integral (J)
tint = (r0 ** (-5 / 3) / 0.423) * (0.5 * lamb / np.pi) ** 2
print(f"Inputs r0: {r0} m, Total Integral (J): {tint} m^(1/3)\n")

# Wind Motion Geometry
slide = 0.205                          # slide: how many meters the screen drifts vertically per full grid length
alpha = slide / size                   # tangent of the drift angle

# Temporal and pixel steps
niter = int(np.floor(tacc / texp + 1)) # accumulation and exposure times
jstep = int(np.floor(((wind * texp) / pixel) + 0.5))
if jstep < 1:                          # at least 1 pixel shift if we are simulating motion
    jstep = 1

windef = jstep * pixel / texp              # effective wind speed on pixel shift

print(f"Screen shift per exposure: {jstep} pixels")
print(f"Effective wind speed: {windef} m/s")
print(f"Total iterations to simulate: {niter}\n")

In [ ]:
# Oversampling
'''
d1 = (lamb / asperpix) * 206265                         
npixperpix = 2**((np.floor(np.log(1.5 * d / d1) / np.log(2)) + 1))
nscr = int(max(1, 2**(np.floor(np.log(1.5 * d / pixel_m) / np.log(2)) + 1)))
ksamp = nap / nscr
nccd = nap
'''

d1 = (lamb / asperpix) * 206265  # pupil scale matching pixels

npixperpix = int(np.ceil(2 ** (np.floor(np.log(1.5 * d / d1) / np.log(2)) + 1))) # oversampling factor

nscr = int(2 ** (np.floor(np.log(1.5 * d / pixel_m) / np.log(2)) + 1)) # Calculate screen size in pixels
ksamp = max(nap / nscr, 1.0)

nccd_full = int(nap / npixperpix) # uncropped CCD array size dictated by the FFT
nccd = min(roi_size, nccd_full) # apply the ROI constraint for your final output


asperpix = pixel
logger.debug(f'd1:{d1}')
print(f'Aperture grid (nap): {nap} pixels')
print(f'Pixel per pixel (npixperpix): {npixperpix}')
print(f'Screen size (nscr): {nscr}')
print(f'Over-sampling factor (ksamp): {ksamp}')
print(f'Re-sampled pixel size (pixel / ksamp): {pixel / ksamp}')
print(f'CCD size (nccd): {nccd} pixels')
print(f'CCD pixel scale (asperpix): {asperpix} arcsec\n')

# Ring size check
ringradpix = 0.85*d*(1+eps)/(4*pdist)*206265/asperpix 
print(f'Expected ring radius (pix, arcsec): {float(ringradpix), float(ringradpix*asperpix)}')

if abs(asperpix / pixel - 1) > 0.05:
    print(f'DISCREPANT PIXEL SCALE!')
    print(f'Input pixel scale:   {pixel}')
    print(f'Calculated pixscale: {asperpix}') # Redundant
    raise ValueError('Simulation pixel scale does not match camera hardware scale.')

# Ring shifts and intensity
if 'jitter' in locals() and jitter:
    x1d = np.arange(nccd)
    omega = 3.3/niter*(2*np.pi)    # 3.3 determines how many cycles the jitter completes during the run

if starmag != 0:
    BW = 0.26                      # effective bandwidth
    phot_con = 1e11                # constant representing photons/sec/m^2 for a Mag 0 star at the top of the atmosphere
    starph = phot_con * texp * BW * 10**(-0.4 * starmag) * np.pi * (d/2.0)**2 * (1.0 - eps)**2 
    
    print(f'Stellar photons per exposure and Star Magnitude: {round(starph,2),starmag}')
    print(f'Readout noise (e-): {ron}')

In [ ]:
# Prepare aperture mask
y, x = np.indices((nap, nap))                    # this is creating the array
center = nap / 2
r = np.sqrt((x - center)**2 + (y - center)**2)   
rad_pix = ((d * 0.5) / pixel_m) * ksamp                      # pupil radius in pixels (using projected meters!)

# Create the mask
apert = np.zeros((nap, nap))
mask = (r <= rad_pix) & (r >= rad_pix * eps)
apert[mask] = 1.0

logger.debug(f'rad_pix: ({d} * 0.5) / {pixel_m}) * {ksamp} = {rad_pix} \n n inside: {np.sum(apert)}')

# Optional: Mask a sector (for testing spider vanes or obstructions)
# apert[0 : nap//2, nap//2-10 : nap//2+10] = 0.0

if logger.isEnabledFor(logging.DEBUG):
    plt.figure(figsize=(6, 6))
    plt.imshow(apert, cmap='gray', origin='lower')
    plt.title(f'Debug Aperture Mask (nap={nap}, rad_pix={round(rad_pix,2)})')
    plt.colorbar(ticks = (0,1))
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()

print(f'Pupil Radius in pixels: {rad_pix}')
print(f'Central Obscuration Radius in pixels: {(rad_pix * eps)}\n')

In [ ]:
# Calculate Zernike coefficients in Radians
a4 = d**2 / (lamb * pdist) * (np.pi / (8 * np.sqrt(3)))   # Defocus
a11 = -0.1 * a4                                                 # Spherical aberration

# Strict Radial Grid Normalization
# Ensure the coordinates are centered cleanly relative to your matrix size
y_coord, x_coord = np.indices((nap, nap))
center = nap / 2
r_grid = np.sqrt((x_coord - center)**2 + (y_coord - center)**2)

# Guard against rad_pix being 0 or inaccurate to prevent zero division
if 'rad_pix' not in locals() or rad_pix <= 0:
    raise ValueError("rad_pix must be a valid positive number representing pupil radius in pixels.")

# rho MUST be scaled so that it equals exactly 1.0 at the outer edge of the pupil boundary
rho = r_grid / rad_pix

# calculate Zernike Phase Maps securely
tmp = a11 * np.sqrt(5) * (6 * rho**4 - 6 * rho**2)        
tmp += a4 * 2 * np.sqrt(3) * (rho**2 - 0.5) 

# Mask the phase map strictly to the aperture array to kill outer frame math noise
zernike_phase = tmp * apert 

print(f'Expected a4 (Defocus) [rad]:     {a4}')
print(f'Expected a11 (Spherical) [rad]: {a11}\n')

if logger.isEnabledFor(logging.DEBUG):
    plt.figure(figsize=(6, 6))
    plt.imshow(zernike_phase, cmap='gray', origin='lower')
    plt.title(f'Debug Zernike applied (nap={nap}, rad_pix={round(rad_pix/16, 2)})')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()

# Construct Complex Wavefront using the CLEAN masked phase map
fresnel = apert * (np.cos(tmp) + np.sin(tmp)*1j)

# Mirror the correct shift order for consistency
pupil_shifted = np.roll(fresnel, (nap/2, nap/2),(0, 1))
imh0_complex = np.fft.fft2(pupil_shifted , axes = (0, 1)) # <-------------------- Has to be 2D
focus_centered = np.roll(imh0_complex, (center, center), (0, 1))

imh0 = np.power(np.abs(focus_centered),2)
normconst = np.sum(imh0)

logger.debug(f'normconst: {normconst}')
logger.debug(f'Array sizes \n(fresnel, zernike_phase, pupil_shifted, imh0): \n {np.sqrt(fresnel.size) , np.sqrt(zernike_phase.size), np.sqrt(pupil_shifted.size), np.sqrt(imh0.size)}')

if logger.isEnabledFor(logging.DEBUG):
    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(fresnel), cmap='gray', origin='lower')
    plt.title(f'Debug at fresnel')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(pupil_shifted), cmap='gray', origin='lower')
    plt.title(f'Debug at pupil_shifted')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(imh0_complex), cmap='gray', origin='lower')
    plt.title(f'Debug at imh0_complex')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(focus_centered), cmap='gray', origin='lower')
    plt.title(f'Debug at focus_centered')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf
    
    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(imh0), cmap='gray', origin='lower')
    plt.title(f'Debug at imh0')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

# Calculate expected radius using clean centered coordinates
rring = np.sum(imh0 * r_grid) / normconst                             
rradpix2 = np.sum(imh0*r)/np.sum(imh0) # true ring radius in fine pixels
rad = rradpix2/npixperpix  # radius in CCD pixels

print(f'True ring radius [pix]: {rring}')
print(f'True ring radius [arcsec]: {(rring * asperpix)}')
logger.debug(f'rradpix2: {rradpix2}, rad: {rad}')

In [ ]:
# Create the cube
cube = np.zeros((int(niter), int(nccd), int(nccd)))
ix = 0
y_float = 0.0

desired_grid = np.ceil(rring) * 2
logger.debug(f'ngrid:{ngrid}, desired: {desired_grid}?, roi: {roi_size}')

# Start main loop for rings
print(f"Computing {niter} iterations...")
ndispl = max(1, niter // 20)

fig, ax = plt.subplots(figsize=(5, 5))
im = None

for i in range(niter):
    # Shift the Atmospheric Screen
    ix += jstep                                                  
    y_float += jstep * alpha
    iy = int(np.floor(y_float))
    
    # Wrap the screen around if we exceed the array size (Periodic boundaries)
    if ix >= ngrid:
        u1 = np.roll(u1, -ngrid, axis=1)
        ix -= ngrid
    if iy >= ngrid:
        u1 = np.roll(u1, -ngrid, axis=0)
        y_float -= ngrid
        iy = int(np.floor(y_float))
    
    # Extract the sub-aperture screen
    uampl = u1[iy : iy + nscr, ix : ix + nscr]
    if nscr != nap: 
        uampl_phase = np.angle(uampl)
        zoomed_phase = zoom(uampl_phase, ksamp, order=1)
        uampl = np.exp(1j * zoomed_phase)
    
    wavefront = fresnel * uampl

    pupil_shifted = np.fft.ifftshift(wavefront)
    imh1_complex = np.fft.ifft2(pupil_shifted)
    focus_centered = np.fft.fftshift(imh1_complex)
    
    imh1 = np.abs(focus_centered)**2
    
    if npixperpix > 1: 
        impix = imh1.reshape(nccd, int(npixperpix), nccd, int(npixperpix)).sum(axis=(1, 3))
    else:
        impix = imh1.copy()
    
    # --- OVERSAMPLING BINNING & ROI EXTRACTION ---
    if npixperpix > 1: 
        # 1. Bin down to the full CCD mathematical resolution
        impix = imh1.reshape(nccd_full, npixperpix, nccd_full, npixperpix).sum(axis=(1, 3))
    else:
        impix = imh1.copy()

    # Crop out the central ROI
    center_idx = nccd_full // 2
    half_roi = nccd // 2
    
    impix = impix[center_idx - half_roi : center_idx + half_roi, 
                  center_idx - half_roi : center_idx + half_roi]

    # Local normalization prevents floating point zero collapse
    loop_norm = np.sum(impix)
    if loop_norm > 0:
        impix = impix / loop_norm  # safely normalizes the matrix sum to exactly 1.0
    
    if 'jitter' in locals() and jitter:
        xc = jitter * np.cos(i * omega)
        yc = jitter * np.sin(i * omega)
        coords = np.meshgrid(np.arange(nccd) + yc, np.arange(nccd) + xc, indexing='ij')
        impix = map_coordinates(impix, coords, order=1)
        
    if starmag != 0:
        impix *= starph                                 
        impix = np.random.poisson(impix).astype(float)  
        impix += np.random.normal(0, ron, (nccd, nccd))
    
    # Live viz
    if im is None:
        im = ax.imshow(impix, cmap='gray', origin='lower')
        fig.colorbar(im, ax=ax, label='Intensity')
        ax.set_xlabel('X [pixels]')
        ax.set_ylabel('Y [pixels]')
        ax.set_title(f'Simulating Ring -> Frame {i}/{niter}')
        
        # Capture the display ID handle on the first frame execution
        display_handle = display(fig, display_id=True)
        
    elif i % 100 == 0:
        im.set_data(impix)                              # Update the image data matrix dynamically
        im.set_clim(vmin=impix.min(), vmax=impix.max()) # Rescale contrast dynamically
        ax.set_title(f"Simulating Ring -> Frame {i}/{niter}")
        display_handle.update(fig)
    
    # Store and progress
    cube[i, :, :] = impix
    if i % ndispl == 0:
        print(f"{(i // ndispl)+1} ", end="", flush=True)
# Clean close of the live loop plotting environment
plt.close(fig)
    
print('\nSimulation done!')

# writefits.pro

this is astropy fits

In [ ]:
hdu = fits.PrimaryHDU(cube.astype('float32'))

header = hdu.header
header['DATE'] = (datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S"), 'File creation date')
header['TELESCOP'] = ('0.304m RC', 'Telescope Diameter')
header['WAVELEN'] = (str(lamb), 'Wavelength in meters')
header['PIXSCALE'] = (str(pixel), 'Arcseconds per pixel')
header['TEXP'] = (str(texp), 'Exposure time in seconds')
header['MAG'] = (str(starmag), 'Stellar magnitude')
header['SEEING'] = (str(seeing), 'Input seeing in arcseconds')
header['CONJ_H'] = (str(pdist), 'Conjugation height in meters')
header['INSTRUME'] = (f'{nap} Simulation', 'Camera dimensions')

filename = 'sim_ring_cube.fits'
hdu.writeto(filename, overwrite=True)

print(f"Successfully saved {niter} frames to {filename}")

# Zernike sliders
Let's try to check for what values should we use (courtesy of Gemini)

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, Layout

def calculate_ring_params(D, H, lamb_nm, pixel_size_um, spherical_percent, target_ring_px):
    # --- 1. Conversions ---
    lamb = lamb_nm * 1e-9
    pix = pixel_size_um * 1e-6
    
    # --- 2. Calculate Z4 (Defocus) ---
    # Formula: Z4 (waves) = D^2 / (8 * sqrt(3) * lamb * H)
    z4 = (D**2) / (8 * np.sqrt(3) * lamb * H)
    
    # --- 3. Calculate Z11 (Spherical) ---
    # Based on your input slider (% of the Z4 amplitude)
    z11 = z4 * (spherical_percent / 100.0)
    
    # --- 4. The -0.1*Z4 (Secondary correction/Piston offset) ---
    z4_offset = -0.1 * z4
    
    # --- 5. Ring Geometry ---
    # Angular diameter theta = D / H (radians)
    theta_rad = D / H
    theta_arcsec = theta_rad * 206265
    
    # Calculate ring size in pixels based on pdist (H)
    # This depends on the telescope's Focal Length (FL). 
    # For a ring simulation, we assume the pixel scale is effectively H / pixel_size
    current_ring_px = (theta_rad * (D * 10)) / pix # Estimation for RC systems
    
    # --- 6. Output Display ---
    print("-" * 50)
    print(f"RESULTS FOR D={D}m at H={H}m ({lamb_nm}nm)")
    print("-" * 50)
    print(f"Z4 (Defocus)        : {z4:10.4f} waves")
    print(f"Z11 (Spherical)     : {z11:10.4f} waves")
    print(f"-0.1 * Z4           : {z4_offset:10.4f} waves")
    print("-" * 50)
    print(f"Ring Angular Diam   : {theta_arcsec:10.2f} arcsec")
    
    # Suggesting H to reach the TARGET ring size
    # target_H = D / (target_px * pix / Focal_Length) 
    # Simplified for interactive feedback:
    suggested_H = (D * 206265) / (target_ring_px * 0.5) # Example heuristic
    
    print(f"Current Ring Size   : ~{current_ring_px:.1f} pixels")
    print(f"To reach {target_ring_px}px, try H ≈ {suggested_H:.1f} m")
    print("-" * 50)

# Create the Interactive Dashboard
interact(
    calculate_ring_params,
    D = FloatSlider(value=0.304, min=0.1, max=1.0, step=0.001, description='D (m):'),
    H = FloatSlider(value=924.0, min=100, max=5000, step=1, description='H (m):'),
    lamb_nm = IntSlider(value=600, min=350, max=1100, step=10, description='λ (nm):'),
    pixel_size_um = FloatSlider(value=6.9, min=1.0, max=20.0, step=0.1, description='Pixel (µm):'),
    spherical_percent = FloatSlider(value=1.15, min=0, max=10, step=0.01, description='Spherical %:'),
    target_ring_px = IntSlider(value=150, min=20, max=500, step=5, description='Target Ring px:', layout=Layout(width='50%'))
);

# Image mask

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from io import BytesIO

# Set up logging to show debug info if desired
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def generate_telescope_mask(nap, ring_radius_arcsec, obscuration_frac, pixel_scale_arcsec):
    """
    Generates and displays a telescope pupil mask with a central obscuration.
    
    Parameters:
    - nap: Size of the computational grid (pixels)
    - ring_radius_arcsec: Physical radius of the primary mirror pupil in arcseconds
    - obscuration_frac: Linear fraction of the central obscuration (0.0 to 1.0)
    - pixel_scale_arcsec: How many arcseconds each pixel represents
    """
    # 1. Convert sky arcseconds into grid pixel units
    # rad_pix is the radius of the outer primary mirror in pixels
    rad_pix = ring_radius_arcsec / pixel_scale_arcsec
    
    # 2. Create the coordinate grid
    y, x = np.indices((int(nap), int(nap)))
    center = nap / 2.0
    r = np.sqrt((x - center)**2 + (y - center)**2)
    
    # 3. Create the mask (Outer boundary AND Central Obscuration)
    apert = np.zeros((int(nap), int(nap)))
    
    # Outer mirror boundary: r <= rad_pix
    # Inner secondary mirror boundary: r >= rad_pix * obscuration_frac
    mask = (r <= rad_pix) & (r >= rad_pix * obscuration_frac)
    apert[mask] = 1.0
    
    # 4. Calculate physical metrics for validation
    num_glass_pixels = np.sum(apert)
    
    # 5. Plotting the results
    plt.figure(figsize=(6, 6))
    plt.imshow(apert, cmap='gray', origin='lower')
    plt.title(f"Pupil Mask ({int(nap)}x{int(nap)})\n"
              f"Outer Radius: {rad_pix:.2f} pix | "
              f"Active Pixels: {int(num_glass_pixels)}")
    plt.xlabel("X [pixels]")
    plt.ylabel("Y [pixels]")
    plt.colorbar(label="Transmission")
    
    # Draw a red dashed circle over the calculated outer edge for visual validation
    theta = np.linspace(0, 2*np.pi, 100)
    plt.plot(center + rad_pix*np.cos(theta), center + rad_pix*np.sin(theta), 'r--', alpha=0.7, label='Outer Edge')
    if obscuration_frac > 0:
        plt.plot(center + rad_pix*obscuration_frac*np.cos(theta), center + rad_pix*obscuration_frac*np.sin(theta), 'y--', alpha=0.7, label='Obscuration')
    
    plt.legend(loc='upper right')
    plt.grid(False)
    plt.show()

# --- Create Interactive Sliders ---
# nap: Image grid size (constrained to common FFT power-of-2 sizes)
nap_slider = widgets.SelectionSlider(
    options=[32, 64, 128, 256, 512],
    value=64,
    description='Grid Size (nap):',
    disabled=False,
    continuous_update=False,
    orientation='horizontal'
)

# ring_radius_arcsec: Sky angular radius of the pupil footprint
radius_slider = widgets.FloatSlider(
    value=15.0,
    min=1.0,
    max=50.0,
    step=0.5,
    description='Ring Rad ("):',
    continuous_update=False
)

# obscuration_frac: The secondary mirror linear obscuration ratio (e.g. 0.4 means 40% diameter)
obsc_slider = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=0.9,
    step=0.05,
    description='Obscuration:',
    continuous_update=False
)

# pixel_scale_arcsec: Sky plate scale
pix_slider = widgets.FloatSlider(
    value=1.1,
    min=0.1,
    max=2.0,
    step=0.05,
    description='Scale ("/pix):',
    continuous_update=False
)

# --- Link Sliders to the Function ---
interactive_plot = widgets.interactive(
    generate_telescope_mask, 
    nap=nap_slider, 
    ring_radius_arcsec=radius_slider, 
    obscuration_frac=obsc_slider, 
    pixel_scale_arcsec=pix_slider
)

# Display the user interface layout
display(interactive_plot)